# LUMENOIR — Kaggle evaluation

This notebook evaluates the existing LUMENOIR pipeline on the permanent Kaggle dataset `/kaggle/input/datasets/chiranth25x/lumenoir-low-light-datasets`, the ExDark mount, the DICM mount, and datasets transferred into `/kaggle/working`.

**Before running**
1. Attach `lumenoir-low-light-datasets` at `/kaggle/input/datasets/chiranth25x/lumenoir-low-light-datasets`.
2. ExDark is at `/kaggle/input/datasets/washingtongold/exdark-dataset`; DICM is at `/kaggle/input/datasets/twozeb/dicm-img`.
3. SICE and NPE/NPE-ex1/NPE-ex2/NPE-ex3, LIME, MEF and VV are not assumed to be permanent Kaggle datasets. Upload/extract them into `/kaggle/working` (or an extracted temporary root) when needed.
4. Attach the LUMENOIR repository and the Zero-DCE++ checkpoint (`zero_dce_plus_plus.pth` or `Epoch99.pth`).
5. Enable a GPU accelerator if you want faster Zero-DCE++ inference; classical stages stay on CPU.

LOL-v1, LOL-v2 Real and LOL-v2 Synthetic are discovered with both train and test pairs; only test/eval records are evaluated. SICE uses the Hugging Face `train/samples/*/{Input,GT}.jpg` and `test/samples/*/{Input,GT}.jpg` layout; only its test samples are evaluated. NPE/NPE-ex1/NPE-ex2/NPE-ex3 are grouped as one logical `npe` dataset (85 images), with subgroup metadata. LIME (10 BMP), MEF (17 PNG), NPE (85) and VV (24) are unpaired, so PSNR/SSIM/LPIPS are never computed for them. JPG, JPEG, PNG and BMP are supported.

The runner never downloads datasets, tunes parameters or fine-tunes the model.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Clone the user-owned joint fork, never the upstream author's repository.
REPO = Path("/kaggle/working/LUMENOIR-JOINT")
if not REPO.is_dir():
    subprocess.run([
        "git", "clone", "--branch", "integration/lumenoir-research",
        "https://github.com/abhi-ironman/LUMENOIR-JOINT.git", str(REPO)
    ], check=True)
else:
    subprocess.run(["git", "fetch", "origin", "integration/lumenoir-research"], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", "integration/lumenoir-research"], cwd=REPO, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", "integration/lumenoir-research"], cwd=REPO, check=True)

sys.path.insert(0, str(REPO))
os.chdir(REPO)
print("Repository:", REPO)
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO, text=True).strip())
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())

In [ ]:
# For full LPIPS/NIQE + paired metrics, install the optional metric stack.
# Kaggle images usually already include PyTorch; this file reuses an installed
# compatible torch version and installs the metric packages if needed.
# !pip install -q -r requirements-metrics.txt
# For classical-only smoke tests, requirements-eval.txt is sufficient.
# !pip install -q -r requirements-eval.txt
print("Runtime dependencies should be installed before running the evaluation cells.")

## 1. Inspect dataset discovery

This writes deterministic manifests under `/kaggle/working/lumenoir-evaluation/manifests`.
It does not load the model or compute metrics.

In [ ]:
from training.kaggle_eval.cli import main

main([
    "--discover-only",
    "--input-root", "/kaggle/input/datasets/chiranth25x/lumenoir-low-light-datasets",
    "--additional-root", "/kaggle/input/datasets/washingtongold/exdark-dataset",
    "--additional-root", "/kaggle/input/datasets/twozeb/dicm-img",
    "--additional-root", "/kaggle/working",
    "--additional-root", "/kaggle/working/lumenoir_additional/extracted",
    "--output-dir", "/kaggle/working/lumenoir-evaluation",
])

## 2. Configure and run

Copy `training/kaggle_eval/example_config.json` to `/kaggle/working/eval_config.json` and edit:

- `additional_roots` for the permanent low-light mount, ExDark, DICM and `/kaggle/working` (where SICE/NPE/LIME/MEF/VV may be transferred);
- dataset `root`, `pairs` and `images` paths; NPE auto-groups from `/kaggle/working`;
- the checkpoint `weights` path;
- `device` (`auto`, `cpu`, `cuda`, `mps`);
- `skip_metrics` if LPIPS/NIQE dependencies are unavailable.

Explicit pair directories are required when auto-discovery is ambiguous. Missing references are skipped and logged; paired metrics are never computed without a ground-truth image. Training records are discovered for counts but are never evaluated.

In [ ]:
import json
from pathlib import Path

from training.kaggle_eval.config import EvalConfig
from training.kaggle_eval.runner import run_evaluation

CONFIG_PATH = Path("/kaggle/working/eval_config.json")
if not CONFIG_PATH.is_file():
    CONFIG_PATH.write_text(
        Path("training/kaggle_eval/example_config.json").read_text(),
        encoding="utf-8",
    )

config = EvalConfig.from_file(CONFIG_PATH)
summary = run_evaluation(config)
print(json.dumps(summary.to_dict(), indent=2))

## 3. Inspect the reports

The aggregate Markdown contains dataset/mode/metric summaries. Per-image JSONL records include metric notes and per-stage timings.

In [ ]:
from pathlib import Path

from IPython.display import Markdown, display

output = Path(config.output_dir)
display(Markdown((output / "aggregate.md").read_text(encoding="utf-8")))

In [ ]:
from pathlib import Path

from IPython.display import Image, display

visuals = sorted((Path(config.output_dir) / "visuals").rglob("*.png"))
print(f"{len(visuals)} before/after images")
for path in visuals[:3]:
    display(Image(filename=str(path)))

## Limitations

- Unpaired datasets (ExDark, DICM, NPE including NPE-ex1/2/3, LIME, MEF, VV) have no ground truth; only no-reference and input-referenced artifact metrics are computed. PSNR/SSIM/LPIPS are never computed for them.
- NPE and its three extensions appear as one `npe` dataset (85 images); `records.jsonl`/`records.csv` retain the `subgroup` and the manifest reports per-subgroup counts.
- SICE is read from `train/samples/*/{Input,GT}.jpg` and `test/samples/*/{Input,GT}.jpg`; only test samples are evaluated, never training samples.
- LOL/SICE training records are discovered for counts but filtered out of the evaluation set.
- SICE Part 1 is Zero-DCE++ pre-training data; paired SICE results are not held out.
- Paired images must have identical dimensions; mismatches are logged as failures.
- LPIPS and NIQE depend on optional packages and their model files; skip them on offline runs.
- No quality improvement should be claimed until these reports are populated with real datasets.

## 4. Joint checkpoint comparison matrix

Attach the two model checkpoints as Kaggle input datasets (do not upload them to GitHub): your friend's fine-tuned/pretrained Zero-DCE++ checkpoint and your LUMENOIR fine-tuned checkpoint. Use separate Kaggle datasets or directories so both files remain unchanged. The example friend path below follows the repository's published-checkpoint config; replace it with the exact fine-tuned checkpoint path if you are comparing fine-tuned vs fine-tuned.

This runs the shared Original/Classical reference once, then Deep Learning and Hybrid for both checkpoints across LOL-v1, LOL-v2 Real and LOL-v2 Synthetic. It uses the same model scale factor (1, matching the fine-tuning config) and a 2×2 Retinex/HueLock matrix. This is an engineering comparison on the selected test splits, not an untouched blind test.

In [ ]:
from pathlib import Path
import torch
from training.kaggle_eval.compare_checkpoints import main as run_joint_matrix

# Update only these two paths to the attached Kaggle input datasets.
# Friend checkpoint: preferably his final fine-tuned .pth; official Epoch99 is acceptable for pretrained-vs-fine-tuned.
FRIEND_CHECKPOINT = Path("/kaggle/input/zero-dce-plus-plus/Epoch99.pth")
ABHI_CHECKPOINT = Path("/kaggle/input/abhi-lumenoir-finetuned/zero_dce_plus_plus.pth")

# Helpful preflight: see every .pth file available to this notebook.
available_checkpoints = sorted(Path("/kaggle/input").rglob("*.pth"))
print("Available checkpoint files:")
for checkpoint_path in available_checkpoints:
    print(" -", checkpoint_path)

missing = [p for p in (FRIEND_CHECKPOINT, ABHI_CHECKPOINT) if not p.is_file()]
if missing:
    raise FileNotFoundError(
        "Attach both checkpoint datasets and update FRIEND_CHECKPOINT / "
        f"ABHI_CHECKPOINT to exact paths. Missing: {missing}"
    )

CONFIG_PATH = Path("/kaggle/working/eval_config.json")
if not CONFIG_PATH.exists():
    CONFIG_PATH.write_text(
        Path("training/kaggle_eval/example_config.json").read_text(encoding="utf-8"),
        encoding="utf-8",
    )

args = [
    "--config", str(CONFIG_PATH),
    "--checkpoint", f"friend={FRIEND_CHECKPOINT}",
    "--checkpoint", f"abhi={ABHI_CHECKPOINT}",
    "--datasets", "lol_v1,lol_v2_real,lol_v2_syn",
    "--model-scale-factor", "1",
    "--retinex-downsample", "2", "3",
    "--hue-anchor", "0", "0.45",
    "--device", "cuda" if torch.cuda.is_available() else "cpu",
    "--latency-warmups", "1",
    "--latency-runs", "3",
    "--output-dir", "/kaggle/working/lumenoir-joint-matrix",
]
exit_code = run_joint_matrix(args)
if exit_code != 0:
    raise RuntimeError(
        "Joint comparison finished with errors. Inspect "
        "/kaggle/working/lumenoir-joint-matrix/matrix_manifest.json."
    )
print("Joint checkpoint comparison completed.")

## 5. Optional CUDA FP32/FP16 parity validation

This is a numerical-agreement check, not a comparison of enhancement quality. It runs only with a Kaggle NVIDIA GPU. It reuses the validated LOL-v2 Real **test** records discovered by the evaluation configuration, then checks the same ten images in classical, deep-learning and hybrid modes for each checkpoint. The results are written under `/kaggle/working/lumenoir-joint-matrix/fp16-parity/`.

Run this cell only after the checkpoint matrix cell above has established `CONFIG_PATH`, `FRIEND_CHECKPOINT` and `ABHI_CHECKPOINT`.

In [ ]:
import subprocess
import torch
from training.kaggle_eval.config import EvalConfig
from training.kaggle_eval.discovery import discover

if not torch.cuda.is_available():
    print("FP32/FP16 parity check skipped: enable a Kaggle NVIDIA GPU to run this CUDA-only validator.")
else:
    # Reuse the same dataset discovery and test-only filtering as the quality matrix.
    parity_config = EvalConfig.from_file(CONFIG_PATH).replace(
        selected_datasets=("lol_v2_real",)
    )
    parity_discovery = discover(parity_config)
    parity_manifest = next(
        (
            item for item in parity_discovery.manifests
            if item.key == "lol_v2_real" and item.present
        ),
        None,
    )
    if parity_manifest is None:
        raise FileNotFoundError(
            "LOL-v2 Real was not discovered. Check the attached dataset input."
        )
    parity_records = parity_manifest.validated_evaluation_records()
    if not parity_records:
        raise RuntimeError("LOL-v2 Real manifest contains no validated test records.")

    # The validator reads a JSON list of absolute image paths.
    parity_image_manifest = Path(
        "/kaggle/working/lol_v2_real_fp16_parity_images.json"
    )
    parity_image_manifest.write_text(
        json.dumps([str(record.low) for record in parity_records[:10]], indent=2),
        encoding="utf-8",
    )
    parity_output = Path(
        "/kaggle/working/lumenoir-joint-matrix/fp16-parity"
    )
    parity_output.mkdir(parents=True, exist_ok=True)

    for label, checkpoint in [
        ("friend", FRIEND_CHECKPOINT),
        ("abhi", ABHI_CHECKPOINT),
    ]:
        output_dir = parity_output / label
        subprocess.run(
            [
                "python", "-m", "training.benchmarks.validate_fp16_parity_gpu",
                "--device", "cuda",
                "--images", str(parity_image_manifest),
                "--weights", str(checkpoint),
                "--count", "10",
                "--modes", "classical,deep_learning,hybrid",
                "--output-dir", str(output_dir),
            ],
            check=True,
        )
        print(f"{label} FP32/FP16 parity reports:", output_dir)
